_If you have already completed this tutorial, you can run the following command to clean up your workspace, then move on to the next chapter_

In [ ]:
rm -rf /scratch/demo_$USER/results *.out

# Introduction to SLURM

SLURM (Simple Linux Utility for Resource Management) is the software used to manage resources on the IFB computing cluster.
It lets you reserve resources and run programs on the cluster's compute nodes.
SLURM commands all start with the letter `s`

## SLURM vocabulary

<img src="images/usine.png" alt="drawing" width="300" style="float: right;"/>

### CPU: the CPU is the smallest unit of a computer processor.
This unit usually corresponds to a thread or a hyperthreaded core.

At the IFB, most nodes have 2 Haswell E5-2695 v3 processors.  
Each processor has 14 physical cores and 28 threads (or virtual cores).  
SLURM therefore counts **56 CPUs** per node.

### RAM: RAM is the memory used by the processor to store the data being analyzed

At the IFB we have on average 252 GB of RAM per node

### Task: a task is a process (the execution of a tool). It can use several CPUs.

### Job: a job is a reservation of resources (CPU, RAM and tasks) to carry out an analysis.

You can picture a job as **a factory** in which you organize the execution of your analysis.

The size of the factory is defined by an amount of CPU, RAM and tasks (the number of CPUs is usually set according to the number of tasks).

<img src="images/chaine.png" alt="drawing" width="300" style="float: right;"/>

### Job step: a “job step” is the part of a job that runs a program

A “job step” can use several “tasks”, but this is specific to certain programs that support parallel computing. In most cases, a job step uses one “task”.

You can picture a job step as **a workshop on the production line** in our factory.


## Mastering SLURM

By using SLURM commands, we will learn how to reserve resources (a factory) and start production lines for our analyses.

The longer our production lines and the more resource-hungry our workshops, the bigger the factories we will need to reserve!

# Submitting a "job"

## Running jobs vs. interactive analysis

There are two ways to use a computing infrastructure:

| Analysis by job submission | Interactive analysis |
| :- | :- |
| Requires writing scripts | The user is at the heart of the analysis process |
| Suited to long and/or repetitive processing | Suited to short, one-off processing |
| Suited to non-interactive tools (no human interaction needed) | Suited to interactive tools (requiring human interaction) |

We are going to learn how to submit jobs.

## The `sbatch` and `srun` commands

A bioinformatics analysis that you want to run on the cluster takes the form of **one or more shell scripts**.

To submit a job to the cluster, we mainly use two SLURM commands:
* `sbatch` reserves resources (the factory) and requests that your script be run
* `srun` launches a “job step” within your script

### The most useful options

`sbatch` and `srun` offer the same basic options:

`--cpus-per-task`: Number of CPUs per task

`--mem`: Amount of RAM allocated per compute node (in MB by default)

`--mem-per-cpu`: Amount of memory expressed per CPU

`--ntasks`: Number of tasks (by default a job is sized with 1 task)

## Getting the example data

We will use 6 sequences to illustrate our SLURM usage examples.

These sequences are available on Zenodo [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.5035749.svg)](https://doi.org/10.5281/zenodo.5035749) but they are already downloaded on the Wave Research cluster.

We will create a scratch folder for this tutorial and copy the demo sequences data into it :

In [ ]:
mkdir -p /scratch/demo_$USER
rsync -ravz /projects/ADVANCED/slurm/data /scratch/demo_$USER/

We will could also have download the data directly from zenodo using `zenodo_get` :

In [ ]:
module load zenodo_get
zenodo_get -o /scratch/demo_$USER/data -t 60 -R 3 10.5281/zenodo.5035749 > /dev/null

<img src="images/petite-chaine.png" alt="drawing" width="100" style="float: right;"/>

## Performing a sequence alignment

We will use two bioinformatics tools to carry out our first analysis with SLURM:
* **bowtie2**: aligns a sequence against a reference genome. An index built from the reference genome is used to perform the alignment
* **samtools**: filters, sorts and converts sequence alignment maps

![Alignment flowchart](images/organigramme-alignement.png)

Each successive processing step requires at least 1 CPU and 1 GB of RAM.

We therefore need to reserve at least 1 task, 1 CPU and 1 GB of RAM for this processing.

The `scripts/alignement_K01_1-v1.sh` script aligns the `K01_1.fastq.gz` file

In [ ]:
cat scripts/alignement_K01_1-v1.sh | pygmentize -f terminal -l bash

We submit the script to the cluster with the sbatch command, specifying our resource needs:

In [ ]:
job_id=`sbatch --parsable scripts/alignement_K01_1-v1.sh`
echo $job_id

Submitting the job created a `slurm-<job_id>.out` file. This file contains the output of the commands run within the job.

(Optional: `--parsable` is only used here to retrieve the `job_id` for later use)

## Good practices for reproducibility

Parameters for `sbatch` can be given:
* on the `sbatch` command line when launching it
* in the header of the `sbatch` script

To make the script easier to reuse and reproduce, it is recommended to declare them directly in the script's preamble:

```bash
#SBATCH --cpus-per-task=10
#SBATCH --mem=1G
```

That way, when you come back to this script days, weeks or months later, you won't need to remember which resources it requires.

## Monitoring your job with `squeue`

The `squeue` command displays information about jobs in the SLURM queue

Check the state of a job:

In [ ]:
squeue -j $job_id

List your user's jobs:

In [ ]:
squeue -u $USER

List your user's running jobs:

In [ ]:
squeue -u $USER -t RUNNING

List pending jobs:

In [ ]:
squeue -t PENDING | head -n 10

## Tracking resource usage with sacct

SLURM includes a mechanism to track the resource consumption of each job.

Jobs that exceed their resource reservation may be killed automatically by the cluster.

The sacct command queries the SLURM database to track resource consumption:

Show basic information about a job: 

In [ ]:
sacct -j $job_id

Show detailed information about a job:

In [ ]:
sacct --format=JobID,JobName,State,Start,Elapsed,CPUTime,NodeList -j $job_id

To list all the available fields, you can use:

In [ ]:
sacct --helpformat

## Cancelling a job

The `scancel` command stops one or more jobs.

To stop the job we launched earlier, use the following command

In [ ]:
scancel $job_id

## Improving the performance of your analysis

<img alt="bowtie2 run time as a function of the number of threads on the K01_1.fastq.gz file" src="images/performances bowtie2.png" width="500px" style="float: right"/>

The `bowtie2` command can use several threads (threads of execution) in parallel:

```
--threads <int> number of alignment threads to launch
```

To speed up the alignment, we can therefore reserve more CPUs and tell `bowtie2` to use them.

The resource allocation for our processing must be adjusted as follows:

![Alignment flowchart 2](images/organigramme-alignement2.png)

We need to reserve at least 1 task, **10 CPUs** and 1 GB of RAM for this processing

The `scripts/alignement_K01_1-v2.sh` script aligns the `K01_1.fastq.gz` file using 10 CPUs:

In [ ]:
cat scripts/alignement_K01_1-v2.sh | pygmentize -f terminal -l bash

In [ ]:
job_id=`sbatch --parsable scripts/alignement_K01_1-v2.sh`
echo $job_id

We can now see that our processing has 10 CPUs:

In [ ]:
sacct --format=JobID,JobName,State,Start,Elapsed,CPUTime,ReqCPUS,NodeList -j $job_id

For every second of real (wall-clock) time, the CPU time is 10 times greater.

## Good practices for reproducibility - continued

SLURM and `sbatch` also provide environment variables you can use within a job to find out what was reserved.  
For instance, the `SLURM_CPUS_PER_TASK` variable gives the number of CPUs per task reserved for this job.

This is very handy for running the `bowtie2` command, for which you must specify the number of threads to use:
```bash
srun bowtie2 --threads="${SLURM_CPUS_PER_TASK}" -x "${reference_index}" -U "${scratch}/data/${file_id}.fastq.gz" -S "${scratch}/results/${file_id}.sam"
```

## More options for `sbatch` and `srun`

|Options|Description|
|:-|:-|
|`--cpus-per-task`|Number of CPUs per task|
|`--mem`|Amount of RAM allocated to the job per node (in MB by default)|
|`--mem-per-cpu`|Amount of memory expressed per CPU|
|`--ntasks`|Number of tasks (by default a job is sized with 1 task)|
|`-J`|Name of the job or job step|
|`--time`|Maximum duration (D-HH:MM:SS)|
|`--output`|Redirection of standard output (may contain the job ID: %j)|
|`--error`|Redirection of standard error (may contain the job ID: %j)|
|`--mail-type`|Enable email notifications (BEGIN, END, FAIL, REQUEUE or ALL)|
|`--mail-user`|Email address for notifications|

Here is an example of a complete script using some of these options:

In [ ]:
cat scripts/alignement_K01_1-v3.sh | pygmentize -f terminal -l bash

# Parallel computing

## How do you align several fastq files in parallel?

In the previous exercise, our job aligned a single fastq file.

To align several fastq files, there are two main strategies:
* Run one "big" job that processes all the files through parallel processes
* Run several small jobs in parallel: each job processes one file

![Parallelization strategies](images/illustration-strategies-parallele.png)

Comparing the two approaches favors the strategy of several small parallel jobs:

|One big job|Several small jobs|
|:-|:-|
|A complex script to write (with a `for` loop) and the use of parallel tasks|A simple script for each job|
|Requires a large resource reservation: all the required resources must be available at the same time|Requires small resource reservations: some alignments can start even if not all resources are immediately available|
|If it crashes, the script must be modified to rerun only the failed alignments|If it crashes, you just rerun the jobs that failed|


## Using a “job array”

SLURM provides a mechanism called a job array that lets you run a similar job several times as a collection (or array) of jobs.

Each job in the array has its own **index**.

The index of the current job is available in the `$SLURM_ARRAY_TASK_ID` environment variable

The `--array` option of `sbatch` is used to launch a collection of jobs

`--array=1,2,3`: submits 3 jobs to the cluster, index 1 for the first job, index 2 for the second and index 3 for the last

`--array=0-15`: submits 16 jobs, with indexes ranging from 0 to 15, index 0 for the first job and 15 for the last

`--array=0-15:2`: defines an index sequence from 0 to 15 with a step of 2, equivalent to --array=0,2,4,6,8,10,12,14

`--array=5-15%4`: defines an index sequence from 5 to 15, but at most 4 jobs run simultaneously

Our `data` folder contains 6 sequence files. We therefore need to create an array of 6 jobs (so 6 index numbers)!

In [ ]:
ls -l /scratch/demo_$USER/data

Bash makes it easy to store the list of files to process in an array variable:

In [ ]:
files=(/scratch/demo_$USER/data/*fastq.gz)

The `files` variable now contains an array where each element holds the path of a different `.fastq.gz` file.

For example, the first element of the files array contains:

In [ ]:
echo ${files[0]}

To keep only the sequence identifier (K01_1), we can use the `basename` command.

The `basename` command returns only the file name from its path.  
The `-s` option also removes a suffix from the file name

In [ ]:
basename -s .fastq.gz ${files[0]}

We can now adapt our alignment script so it can be run multiple times as part of a job array.

In [ ]:
cat scripts/alignement.sh | pygmentize -f terminal -l bash

The job will run 6 times (`--array=0-5`) and each run will process a different file based on the current job's index (`file_id=$(basename -s .fastq.gz "${files[$SLURM_ARRAY_TASK_ID]}")`).

![Job array flowchart](images/organigramme-job-array.png)

In [ ]:
job_id=`sbatch --parsable scripts/alignement.sh`
echo $job_id

In [ ]:
sacct --format=JobID%15,JobName%15,State,Start,Elapsed,CPUTime,ReqCPUS,NodeList -j $job_id

### SLURM job efficiency

https://ifb-elixirfr.gitlab.io/cluster/doc/slurm/slurm_efficiency/ 

`reportseff` takes a `jobid` and reports on how efficiently the CPU and memory were used.

In [ ]:
module load reportseff

reportseff $job_id